In [15]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    OneHotEncoder, OrdinalEncoder, LabelEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
import joblib
import os
from datetime import datetime

In [2]:
#modular pipeline architecture for heart disease predication
print("=" * 80)
print("Heart Disease Predication - Modular Pipeline Architecture")
print("=" * 80)

Heart Disease Predication - Modular Pipeline Architecture


In [3]:
#1. Load dataset
print("\n" + "=" * 80)
print("1. data loading")
print("=" * 80)

# Load the engineered dataset (or create a sample if not available)
try:
    df = pd.read_csv("../Data/processed/heart_disease_engineered.csv")
    print("loaded engineered dataset")
except:
    # Create sample data for demonstration
    print("Engineered dataset not found, creating sample data")
    np.random.seed(42)
    n_samples = 1000

print(f"dataset shape: {df.shape}")
print(f"columns: {df.columns.tolist()}")


1. data loading
loaded engineered dataset
dataset shape: (180, 48)
columns: ['patient_id', 'slope_of_peak_exercise_st_segment', 'thal', 'resting_blood_pressure', 'chest_pain_type', 'num_major_vessels', 'fasting_blood_sugar_gt_120_mg_per_dl', 'resting_ekg_results', 'serum_cholesterol_mg_per_dl', 'oldpeak_eq_st_depression', 'sex', 'age', 'max_heart_rate_achieved', 'exercise_induced_angina', 'heart_disease_present', 'age_group', 'is_elderly', 'age_decade', 'bp_category', 'has_hypertension', 'bp_severity', 'cholesterol_category', 'high_cholesterol', 'cholesterol_risk', 'predicted_max_hr', 'hr_reserve', 'inadquate_hr_response', 'hr_category', 'simple_risk_score', 'metabolic_score', 'possible_metabolic_syndrome', 'exercise_stress_score', 'age_cholesterol_interaction', 'bp_age_interaction', 'st_angina_interaction', 'high_risk_male', 'age_squared', 'cholesterol_squared', 'bp_squared', 'log_cholesterol', 'log_oldpeak', 'age_quintile', 'resting_blood_pressure_quintile', 'serum_cholesterol_mg_pe

In [4]:
# Feature Categorisation
print("\n" + "=" * 80)
print("2. feature categorization")
print("=" * 80)

# exclude identifiers and target
excluded_cols = ['patient_id', 'heart_disease_present']

# define feature groups based on your project analysis
numerical_features = [
    'age', 'resting_blood_pressure', 'serum_cholesterol_mg_per_dl',
    'max_heart_rate_achieved', 'oldpeak_eq_st_depression', 'num_major_vessels'
]

categorical_features = [
    'thal', 'chest_pain_type', 'resting_ekg_results',
    'slope_of_peak_exercise_st_segment'
]

binary_features = [
    'sex', 'exercise_induced_angina', 'fasting_blood_sugar_gt_120_mg_per_dl'
]

# identify engineered features (any features not in the above categories)
all_features = [col for col in df.columns if col not in excluded_cols]
engineered_features = [
    col for col in all_features 
    if col not in numerical_features + categorical_features + binary_features
]

print(f"numerical features ({len(numerical_features)}): {numerical_features}")
print(f"categorical features ({len(categorical_features)}): {categorical_features}")
print(f"binary features ({len(binary_features)}): {binary_features}")
print(f"engineered features ({len(engineered_features)}): {engineered_features}")



2. feature categorization
numerical features (6): ['age', 'resting_blood_pressure', 'serum_cholesterol_mg_per_dl', 'max_heart_rate_achieved', 'oldpeak_eq_st_depression', 'num_major_vessels']
categorical features (4): ['thal', 'chest_pain_type', 'resting_ekg_results', 'slope_of_peak_exercise_st_segment']
binary features (3): ['sex', 'exercise_induced_angina', 'fasting_blood_sugar_gt_120_mg_per_dl']
engineered features (33): ['age_group', 'is_elderly', 'age_decade', 'bp_category', 'has_hypertension', 'bp_severity', 'cholesterol_category', 'high_cholesterol', 'cholesterol_risk', 'predicted_max_hr', 'hr_reserve', 'inadquate_hr_response', 'hr_category', 'simple_risk_score', 'metabolic_score', 'possible_metabolic_syndrome', 'exercise_stress_score', 'age_cholesterol_interaction', 'bp_age_interaction', 'st_angina_interaction', 'high_risk_male', 'age_squared', 'cholesterol_squared', 'bp_squared', 'log_cholesterol', 'log_oldpeak', 'age_quintile', 'resting_blood_pressure_quintile', 'serum_choles

In [5]:
#3. create modular pipepline
print("\n" + "=" * 80)
print("3. create modular pipelines")
print("=" * 80)

# numerical pipeline
print("\nCreating a numerical Pipeline")
# define which scaler to use for each numerical feature (based on your scaling strategy)
# this should come from your scaling documentation
numerical_scalers = {
    'age': 'minmax',
    'resting_blood_pressure': 'standard',
    'serum_cholesterol_mg_per_dl': 'standard',
    'max_heart_rate_achieved': 'standard',
    'oldpeak_eq_st_depression': 'robust',
    'num_major_vessels': 'minmax'
}

# separate features by scaler type
standard_features = [f for f in numerical_features if numerical_scalers.get(f) == 'standard']
minmax_features = [f for f in numerical_features if numerical_scalers.get(f) == 'minmax']
robust_features = [f for f in numerical_features if numerical_scalers.get(f) == 'robust']

print(f"standardscaler: {standard_features}")
print(f"minmaxscaler: {minmax_features}")
print(f"robustscaler: {robust_features}")

# create numerical pipeline with different scalers
numerical_pipeline = ColumnTransformer([
    ('standard', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), standard_features),
    
    ('minmax', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', MinMaxScaler())
    ]), minmax_features),
    
    ('robust', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler())
    ]), robust_features)
])


3. create modular pipelines

Creating a numerical Pipeline
standardscaler: ['resting_blood_pressure', 'serum_cholesterol_mg_per_dl', 'max_heart_rate_achieved']
minmaxscaler: ['age', 'num_major_vessels']
robustscaler: ['oldpeak_eq_st_depression']


In [30]:
print("\nCreating categorical pipeline...")

# define encoding strategies for each categorical feature
categorical_encoding = {
    'thal': 'onehot',  # no ordinal relationship
    'chest_pain_type': 'ordinal',  # has severity order
    'resting_ekg_results': 'onehot',  # categorical
    'slope_of_peak_exercise_st_segment': 'ordinal'  # has order (1=upsloping, 2=flat, 3=downsloping)
}

# separate features by encoding type
onehot_features = [f for f in categorical_features if categorical_encoding.get(f) == 'onehot']
ordinal_features = [f for f in categorical_features if categorical_encoding.get(f) == 'ordinal']

print(f"  onehot encoding: {onehot_features}")
print(f"  ordinal encoding: {ordinal_features}")

# FIX: Define categories properly - using flat lists, not nested lists
ordinal_categories_dict = {
    'chest_pain_type': [1, 2, 3, 4],  # FIXED: flat list, not [[1, 2, 3, 4]]
    'slope_of_peak_exercise_st_segment': [1, 2, 3]  # FIXED: flat list, not [[1, 2, 3]]
}

# Build categories list for OrdinalEncoder
categories_list = []
for feat in ordinal_features:
    if feat in ordinal_categories_dict:
        categories_list.append(ordinal_categories_dict[feat])
    else:
        # For other ordinal features, get unique values from data
        unique_vals = df[feat].dropna().unique()
        # Sort if possible (for numeric values)
        try:
            sorted_vals = sorted(unique_vals)
            categories_list.append(sorted_vals)
        except:
            categories_list.append(list(unique_vals))

# FIX: Create categorical pipeline with proper OrdinalEncoder
categorical_pipeline = ColumnTransformer([
    ('onehot', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
    ]), onehot_features),
    
    ('ordinal', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(
            categories=categories_list,  # FIXED: Use the properly built list
            handle_unknown='use_encoded_value',  # FIXED: Add this parameter
            unknown_value=-1  # FIXED: Add this parameter
        ))
    ]), ordinal_features)
])


Creating categorical pipeline...
  onehot encoding: ['thal', 'resting_ekg_results']
  ordinal encoding: ['chest_pain_type', 'slope_of_peak_exercise_st_segment']


In [25]:
#Binary pipeline
print("\nCreating binary pipeline...")

# binary features typically need minimal processing
# just ensure they're 0/1 and handle any missing values
binary_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    # optional: could add a scaler if needed, but usually binary stays as-is
])


Creating binary pipeline...


In [26]:
#Engineered Features pipeline
print("\nCreating engineered features pipeline...")

# engineered features need different handling based on their type
# first, categorize engineered features
if engineered_features:
    engineered_numerical = [f for f in engineered_features if pd.api.types.is_numeric_dtype(df[f])]
    engineered_categorical = [f for f in engineered_features if not pd.api.types.is_numeric_dtype(df[f])]
    
    print(f"  numerical engineered: {engineered_numerical}")
    print(f"  categorical engineered: {engineered_categorical}")
    
    # create pipeline for engineered features
    engineered_pipeline = ColumnTransformer([
        ('engineered_numerical', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())  # default for engineered numerical
        ]), engineered_numerical),
        
        ('engineered_categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
        ]), engineered_categorical)
    ])
else:
    print("  no engineered features found")
    engineered_pipeline = 'passthrough'


Creating engineered features pipeline...
  numerical engineered: ['is_elderly', 'age_decade', 'has_hypertension', 'bp_severity', 'high_cholesterol', 'cholesterol_risk', 'predicted_max_hr', 'hr_reserve', 'inadquate_hr_response', 'simple_risk_score', 'metabolic_score', 'possible_metabolic_syndrome', 'exercise_stress_score', 'age_cholesterol_interaction', 'bp_age_interaction', 'st_angina_interaction', 'high_risk_male', 'age_squared', 'cholesterol_squared', 'bp_squared', 'log_cholesterol', 'log_oldpeak', 'age_quintile', 'resting_blood_pressure_quintile', 'serum_cholesterol_mg_per_dl_quintile', 'max_heart_rate_achieved_quintile', 'cholesterol_age_ratio', 'bp_age_ratio', 'hr_bp_ratio']
  categorical engineered: ['age_group', 'bp_category', 'cholesterol_category', 'hr_category']


In [27]:
# Combine into master pipeline
print("\n" + "=" * 80)
print("4. combine into master pipeline")
print("=" * 80)

# create the master column transformer
master_transformer = ColumnTransformer([
    ('numerical', numerical_pipeline, numerical_features),
    ('categorical', categorical_pipeline, categorical_features),
    ('binary', binary_pipeline, binary_features),
    ('engineered', engineered_pipeline, engineered_features) if engineered_features else ('passthrough', 'passthrough', [])
], remainder='drop')  # drop any unprocessed columns

print("created master column transformer")
print(f"total feature groups: 4")
print(f"total features processed: {len(numerical_features + categorical_features + binary_features + engineered_features)}")


4. combine into master pipeline
created master column transformer
total feature groups: 4
total features processed: 46


In [28]:
# Add feature selection and dimensionality reduction
print("\n" + "=" * 80)
print("5. add feature selection and dimensionality reduction")
print("=" * 80)

# create the complete preprocessing pipeline
preprocessing_pipeline = Pipeline([
    ('transformer', master_transformer),
    ('feature_selection', SelectKBest(score_func=f_classif, k='all')),  # will be tuned
    ('dimensionality_reduction', PCA(n_components=0.95))  # keep 95% variance
])

print("created complete preprocessing pipeline")
print("pipeline steps:")
print("  1. column transformer (feature-specific processing)")
print("  2. feature selection (selectkbest with f-classif)")
print("  3. dimensionality reduction (pca keeping 95% variance)")


5. add feature selection and dimensionality reduction
created complete preprocessing pipeline
pipeline steps:
  1. column transformer (feature-specific processing)
  2. feature selection (selectkbest with f-classif)
  3. dimensionality reduction (pca keeping 95% variance)


In [29]:
# Create model pipelines
print("\n" + "=" * 80)
print("6. create model pipelines")
print("=" * 80)

# model 1: logistic regression
logistic_pipeline = Pipeline([
    ('preprocessing', preprocessing_pipeline),
    ('classifier', LogisticRegression(
        random_state=42,
        max_iter=1000,
        class_weight='balanced'
    ))
])

# model 2: random forest
random_forest_pipeline = Pipeline([
    ('preprocessing', preprocessing_pipeline),
    ('classifier', RandomForestClassifier(
        random_state=42,
        n_estimators=100,
        class_weight='balanced',
        n_jobs=-1
    ))
])

# model 3: support vector machine
svm_pipeline = Pipeline([
    ('preprocessing', preprocessing_pipeline),
    ('classifier', SVC(
        random_state=42,
        probability=True,
        class_weight='balanced'
    ))
])

# model 4: xgboost
xgb_pipeline = Pipeline([
    ('preprocessing', preprocessing_pipeline),
    ('classifier', XGBClassifier(
        random_state=42,
        n_estimators=100,
        use_label_encoder=False,
        eval_metric='logloss',
        n_jobs=-1
    ))
])

print("created 4 model pipelines:")
print("  1. logistic regression (interpretable baseline)")
print("  2. random forest (handles non-linearity)")
print("  3. support vector machine (good for high dimensions)")
print("  4. xgboost (state-of-the-art for tabular data)")


6. create model pipelines
created 4 model pipelines:
  1. logistic regression (interpretable baseline)
  2. random forest (handles non-linearity)
  3. support vector machine (good for high dimensions)
  4. xgboost (state-of-the-art for tabular data)


In [31]:
print("="*80)
print("7. create pipeline for training and inference")
print("="*80)

class HeartDiseasePipeline:
    def __init__(self, model_type='xgboost'):
        """
        Comprehensive pipeline class for heart disease prediction
        
        Args:
            model_type (str): 'logistic', 'randomforest', 'svm', or 'xgboost'
        """
        self.model_type = model_type
        self.pipeline = None
        self.feature_names = None
        self.is_fitted = False
        
        # Select appropriate pipeline
        if model_type == 'logistic':
            self.pipeline = logistic_pipeline  # FIXED: was logisticpipeline
        elif model_type == 'randomforest':
            self.pipeline = random_forest_pipeline  # FIXED: was randomforestpipeline
        elif model_type == 'svm':
            self.pipeline = svm_pipeline  # FIXED: was svmpipeline
        elif model_type == 'xgboost':
            self.pipeline = xgb_pipeline
        else:
            raise ValueError("model_type must be one of: 'logistic', 'randomforest', 'svm', 'xgboost'")
    
    def fit(self, X, y):
        """Fit the pipeline with training data"""
        self.pipeline.fit(X, y)
        self.is_fitted = True
        self.feature_names = X.columns.tolist()
        print(f"Fitted {self.model_type} pipeline successfully!")
        return self
    
    def predict(self, X):
        """Predict heart disease presence"""
        if not self.is_fitted:
            raise ValueError("Pipeline must be fitted before prediction")
        return self.pipeline.predict(X)
    
    def predict_proba(self, X):
        """Predict probability of heart disease"""
        if not self.is_fitted:
            raise ValueError("Pipeline must be fitted before prediction")
        return self.pipeline.predict_proba(X)
    
    def get_feature_importance(self):
        """Extract feature importance (works for tree-based models)"""
        if not self.is_fitted:
            raise ValueError("Pipeline must be fitted before extracting importance")
            
        # Get the classifier from the pipeline
        classifier = self.pipeline.named_steps['classifier']
        
        if hasattr(classifier, 'feature_importances_'):
            # Tree-based models (RandomForest, XGBoost)
            importances = classifier.feature_importances_
        elif hasattr(classifier, 'coef_'):
            # Linear models (Logistic Regression)
            importances = np.abs(classifier.coef_[0])
        else:
            raise ValueError("Feature importance not available for this model")
        
        # Get feature names after preprocessing
        feature_names = self.pipeline[:-1].get_feature_names_out()
        
        importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': importances
        }).sort_values('importance', ascending=False)
        
        return importance_df
    
    def save(self, filepath):
        """Save the fitted pipeline"""
        if not self.is_fitted:
            raise ValueError("Pipeline must be fitted before saving")
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"{self.model_type}_pipeline_{timestamp}.pkl"
        full_path = os.path.join(filepath, filename)
        
        joblib.dump(self.pipeline, full_path)
        print(f"Pipeline saved to: {full_path}")
        return full_path
    
    def __repr__(self):
        return f"HeartDiseasePipeline(model_type='{self.model_type}', fitted={self.is_fitted})"

# Test the pipeline class [file:1]
print("\nTesting Pipeline Class...")
X = df.drop(['patient_id', 'heart_disease_present'], axis=1)
y = df['heart_disease_present']

# Split for testing
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Create and test pipeline
hd_pipeline = HeartDiseasePipeline(model_type='xgboost')
hd_pipeline.fit(X_train, y_train)

# Test predictions
train_pred = hd_pipeline.predict(X_train)
test_pred = hd_pipeline.predict(X_test)
train_proba = hd_pipeline.predict_proba(X_train)

print(f"Training predictions shape: {train_pred.shape}")
print(f"Test predictions shape: {test_pred.shape}")
print(f"Sample predictions: {test_pred[:5]}")
print(f"Sample probabilities: {train_proba[:5, 1]}")  # Probability of heart disease


7. create pipeline for training and inference

Testing Pipeline Class...
Fitted xgboost pipeline successfully!
Training predictions shape: (144,)
Test predictions shape: (36,)
Sample predictions: [1 1 1 0 0]
Sample probabilities: [0.00144405 0.00808488 0.01314357 0.9782694  0.00236567]


c:\Users\rache\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [13:07:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [32]:
#Test the Complete Pipeline
print("="*80)
print("8. test the pipeline")
print("="*80)

# Comprehensive pipeline testing [file:1]
print("Pipeline Testing Results:")
print("-" * 50)

# 1. Verify preprocessing works
X_processed = hd_pipeline.pipeline[:-1].fit_transform(X_train, y_train)
print(f"✓ Preprocessing output shape: {X_processed.shape}")

# 2. Verify predictions
test_proba = hd_pipeline.predict_proba(X_test)
print(f"✓ Test predictions generated: {len(test_proba)} samples")
print(f"✓ Positive class probability range: {test_proba[:,1].min():.3f} - {test_proba[:,1].max():.3f}")

# 3. Basic model performance
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

test_accuracy = accuracy_score(y_test, test_pred)
test_auc = roc_auc_score(y_test, test_proba[:,1])

print(f"✓ Test Accuracy: {test_accuracy:.3f}")
print(f"✓ Test AUC-ROC: {test_auc:.3f}")

print("\nClassification Report:")
print(classification_report(y_test, test_pred))

# 4. Feature importance
importance_df = hd_pipeline.get_feature_importance()
print("\nTop 10 Most Important Features:")
print(importance_df.head(10).to_string(index=False))


8. test the pipeline
Pipeline Testing Results:
--------------------------------------------------
✓ Preprocessing output shape: (144, 18)
✓ Test predictions generated: 36 samples
✓ Positive class probability range: 0.001 - 0.999
✓ Test Accuracy: 0.778
✓ Test AUC-ROC: 0.884

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.70      0.78        20
           1       0.70      0.88      0.78        16

    accuracy                           0.78        36
   macro avg       0.79      0.79      0.78        36
weighted avg       0.80      0.78      0.78        36


Top 10 Most Important Features:
feature  importance
   pca3    0.131467
  pca10    0.105644
   pca0    0.091378
   pca7    0.079956
  pca17    0.071845
   pca6    0.069663
   pca2    0.065997
  pca15    0.058960
   pca5    0.054000
   pca4    0.043574


In [35]:
# Save Pipeline Architecture
print("="*80)
print("9. save pipeline architecture")
print("="*80)

# Create output directory [file:1]
output_dir = "saved_pipelines"
os.makedirs(output_dir, exist_ok=True)

# Save all individual pipeline components
components_to_save = {
    'preprocessing_pipeline': preprocessing_pipeline,
    'logistic_pipeline': logistic_pipeline,
    'randomforest_pipeline': random_forest_pipeline,
    'svm_pipeline': svm_pipeline,
    'xgb_pipeline': xgb_pipeline,
    'fitted_xgb_pipeline': hd_pipeline.pipeline,  # The fully trained one
    'feature_importance': importance_df
}

for name, component in components_to_save.items():
    filepath = os.path.join(output_dir, f"{name}.pkl")
    joblib.dump(component, filepath)
    print(f"✓ Saved: {filepath}")

# Save the complete HeartDiseasePipeline instance
hd_pipeline.save(output_dir)

print(f"\n✓ All components saved to '{output_dir}' directory")
print("✓ Ready for production deployment!")


9. save pipeline architecture
✓ Saved: saved_pipelines\preprocessing_pipeline.pkl
✓ Saved: saved_pipelines\logistic_pipeline.pkl
✓ Saved: saved_pipelines\randomforest_pipeline.pkl
✓ Saved: saved_pipelines\svm_pipeline.pkl
✓ Saved: saved_pipelines\xgb_pipeline.pkl
✓ Saved: saved_pipelines\fitted_xgb_pipeline.pkl
✓ Saved: saved_pipelines\feature_importance.pkl
Pipeline saved to: saved_pipelines\xgboost_pipeline_20251229_130903.pkl

✓ All components saved to 'saved_pipelines' directory
✓ Ready for production deployment!


In [ ]:
# Create Configuration File
print("="*80)
print("10. create configuration file")
print("="*80)

# Comprehensive configuration documentation [file:1]
config = {
    "project": "Heart Disease Prediction - Modular Pipeline",
    "version": "1.0",
    "timestamp": datetime.now().isoformat(),
    
    "feature_groups": {
        "numerical_features": numerical_features, 
        "numerical_scalers": {
            "standard": standard_features, 
            "minmax": minmax_features, 
            "robust": robust_features 
        },
        "categorical_features": categorical_features, 
        "categorical_encoding": {
            "onehot": onehot_features,
            "ordinal": ordinal_features 
        },
        "binary_features": binary_features,
        "engineered_features": engineered_features if 'engineered_features' in locals() else []
    },
    
    "preprocessing": {
        "imputation": {
            "numerical": "median",
            "categorical": "most_frequent", 
            "binary": "most_frequent"
        },
        "feature_selection": "SelectKBest(f_classif, k=all)",  # Tune k later
        "dimensionality_reduction": "PCA(n_components=0.95)"
    },
    
    "models": {
        "logistic": {
            "class_weight": "balanced",
            "max_iter": 1000,
            "random_state": 42
        },
        "randomforest": {
            "n_estimators": 100,
            "class_weight": "balanced", 
            "n_jobs": -1,
            "random_state": 42
        },
        "svm": {
            "probability": True,
            "class_weight": "balanced",
            "random_state": 42
        },
        "xgboost": {
            "n_estimators": 100,
            "random_state": 42,
            "n_jobs": -1,
            "eval_metric": "logloss"
        }
    },
    
    "target": {
        "name": "heart_disease_present",
        "positive_class": 1,
        "interpretation": "1=Heart Disease Present, 0=Absent"
    },
    
    "deployment": {
        "saved_directory": output_dir,
        "usage": "from joblib import load\npipeline = load('fitted_xgb_pipeline.pkl')\npredictions = pipeline.predict(new_data)"
    }
}

# Save configuration
config_path = os.path.join(output_dir, "pipeline_config.json")
with open(config_path, 'w') as f:
    import json
    # Convert lists to strings for JSON serialization
    config_json = json.dumps(config, indent=2, default=str)
    f.write(config_json)

print(f"✓ Configuration saved: {config_path}")
print("\nPipeline is production-ready!")
print("="*80)

10. create configuration file
✓ Configuration saved: saved_pipelines\pipeline_config.json

Pipeline is production-ready!
